# பாடம் 11 - மாடல் சூழல் நெறிமுறை (MCP)

**Model Context Protocol (MCP)** என்பது ஒரு திறந்த தரநிலை ஆகும், இது முகவர்கள் இயங்கும் நேரத்தில் கருவிகள், வளங்கள் மற்றும் தரவுத் தளங்களை தானாக கண்டறிந்து பயன்படுத்த அனுமதிக்கிறது. ஒரு முகவரில் கருவிகளை கடுமையாக நிரலாக்குவதற்குப் பதிலாக, MCP முகவர்களுக்கு தேவையான போது திறன்களை வெளிப்படுத்தும் வெளிப்புற சர்வர்களுடன் இணைக்க அனுமதிக்கிறது.

இந்த பாடத்தில், நீங்கள் கற்றுக்கொள்வீர்கள்:
- MCP என்ன மற்றும் அது முகவர் அமைப்புகளுக்கு ஏன் முக்கியமானது
- MCP கிளையன்ட்-சர்வர் கட்டமைப்பு எவ்வாறு செயல்படுகிறது
- MCP முறை கருவி கண்டறிதலைப் பயன்படுத்தும் முகவர்களை எப்படி உருவாக்குவது


## அமைப்பு

**முன் நிபந்தனைகள்:**
- வெளியிடப்பட்ட மாடலை கொண்ட Azure AI Foundry திட்டம்
- அங்கீகாரத்திற்காக `az login` இயக்கவும்


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) என்பது என்ன?

MCP என்பது செயற்கை நுண்ணறிவு ஏஜெண்டுகள் வெளிப்புற கருவிகள் மற்றும் தரவுத் ஆதாரங்களை கண்டறிந்து அவற்றுடன் தொடர்பு கொள்ள ஒரு நிலையான முறையை வரையறுக்கிறது:

- **MCP Server**: ஒரு நிலையான புரோட்டோக்கால் வழியாக கருவிகள், வளங்கள் மற்றும் ப்ராம்டுகளை வெளிப்படுத்துகிறது
- **MCP Client**: சேவையகங்களுக்கு இணைந்து கிடைக்கும் திறன்களை கண்டறியும் ஏஜெண்ட் இயங்கு சூழல்
- **Dynamic Discovery**: ஏஜெண்ட்களுக்கு முன்கூட்டியே கடுமையாக குறியாக்கப்பட்ட கருவிகள் அவசியமில்லை — அவை இயங்கும் போது கிடைக்கும் வளங்களை தானாகவே கண்டறிகின்றன

இது விரிவாக்கக்கூடிய ஏஜெண்ட் அமைப்புகளை உருவாக்க மிகவும் சக்திவாய்ந்தது, அங்கு புதிய திறன்களை ஏஜெண்ட் கோடுகளை மாற்றாமல் சேர்க்க முடியும்.


## MCP எவ்வாறு செயல்படுகிறது

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. முகவர் (MCP கிளையენტი) ஒரு MCP சேவையகத்துடன் இணைகிறது
2. சேவையகம் கிடைக்கும் கருவிகள் மற்றும் அவற்றின் வடிவமைப்புகளின் பட்டியலுடன் பதிலளிக்கிறது
3. பின்னர் முகவர் தன் காரணித்தல் செயல்முறையின் போது கண்டறியப்பட்ட எந்த கருவியையும் அழைக்கலாம்
4. முடிவுகள் ஒரே நெறிமுறையின் மூலம் மீண்டும் திரும்பி வருகின்றன


## MCP கருவி கண்டறிதலை சிமுலேட் செய்தல்

உண்மையான MCP சர்வர் ஓடிக்கொண்டிருக்கும் சர்வர் செயல்முறை (ஓடிக்கொண்டிருக்கும் சர்வர் செயல்முறை) தேவைப்படுவதால், MCP-இன் இணைக்கப்பட்ட வசதி சேவை வழங்கும் அம்சங்களை சிமுலேட் செய்ய `@tool` செயல்பாடுகளைப் பயன்படுத்தி இந்த மாதிரியை நாம் விளக்கப் போகிறோம்.

உற்பத்தியில், இந்த கருவிகள் உள்ளூரில் வரையறுக்கப்படுவதற்கு பதிலாக MCP சர்வர் மூலம் தானாகவே கண்டறியப்படுவார்கள்.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-பாணி கருவிகளுடன் ஒரு ஏஜென்டை உருவாக்குதல்


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## உற்பத்தி சூழலில் MCP

உற்பத்தி சூழலில், MCP பல சக்திவாய்ந்த நடைமுறைகளை சாதிக்க உதவுகிறது:

- **கருவிகளை தானாக கண்டறிதல்**: ஏஜென்டுகள் MCP சேவையகங்களுக்கு இணைந்து இயங்கும் நேரத்தில் கருவிகளை கண்டறிகிறார்கள்
- **பிரிக்கப்பட்ட கட்டமைப்பு**: கருவி வழங்குநர்கள் ஏஜென்டுகளைக் சாராமல் தாங்களே புதுப்பிப்புகளை செய்ய முடியும்
- **அமைப்புகளுக்கு மத்தியில் பகிர்வு**: அணிகள் MCP சேவையகங்களின் மூலம் எந்த ஏஜென்டும் பயன்படுத்தக்கூடிய திறன்களை வெளிப்படுத்த முடியும்
- **Microsoft Agent Framework ஆதரவு**: MAF இல் உள்ளகமாக MCP கிளையன்ட் ஆதரவு `mcp` ஒருங்கிணைப்பின் மூலம் உள்ளது

உண்மையான MCP சேவையகத்தை MAF உடன் பயன்படுத்த, நீங்கள் `hosted_mcp_tool()` அல்லது MCP கிளையன்ட் ஒருங்கிணைப்பின் மூலம் இணைக்கலாம்.

**மேலும் அறிய:**
- [MCP விவரக்குறிப்பு](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP ஆதரவு](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## சுருக்கம்

இந்த பாடத்தில், நீங்கள் கற்றுக்கொண்டவை:
- **MCP** என்பது ஏஜென்ட்களுக்கும் கருவி வழங்குநர்களுக்கும் இடையிலான மாறுபடக்கூடிய கருவி கண்டறிதலுக்கான ஒரு திறந்த தரநிலை ஆகும்
- **கிளையன்ட்-சர்வர் கட்டமைப்பு** ஏஜென்ட்களுக்கு இயக்கநேரத்தில் திறன்களை கண்டறிய அனுமதிக்கிறது
- MCP **விரிவாக்கக்கூடிய, தனியாக பிரிக்கப்பட்ட ஏஜென்ட் அமைப்புகளை** சாத்தியப்படுத்துகிறது, இதில் குறியீட்டில் மாற்றங்கள் செய்யாமல் கருவிகளைச் சேர்க்கலாம்
- Microsoft Agent Framework உற்பத்தி பயன்பாட்டிற்காக **உள்ளமைந்த MCP ஆதரவைக்** வழங்குகிறது


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
மறுப்பு அறிவிப்பு:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையான Co-op Translator (https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. আমরা তுல்லியத்திற்காக முயற்சி செய்தாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான தகவல்கள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனத்தில் கொள்ளுங்கள். மூல ஆவணம் அதன் இயல்பான மொழியிலேயே அதிகாரபூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு தொழிற்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்படும் எந்தவொரு புரிதல்தவறுகளுக்கும் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பேற்றுக்கொள்ளமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
